In [ ]:

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=False
# )

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

# tokenizer = AutoTokenizer.from_pretrained("EleutherAI/llemma_7b")
# model = AutoModelForCausalLM.from_pretrained("EleutherAI/llemma_7b", quantization_config=quant_config, device_map={"": 0})


# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quant_config,
#                                              device_map={"": 0})


tokenizer = AutoTokenizer.from_pretrained('deepseek-ai/deepseek-math-7b-base')
model = AutoModelForCausalLM.from_pretrained('deepseek-ai/deepseek-math-7b-base', quantization_config=quant_config,
                                             device_map='auto')


# filename = 'deepseek-math-7b-rl.Q8_0.gguf'
# tokenizer = AutoTokenizer.from_pretrained('QuantFactory/deepseek-math-7b-rl-GGUF')#, gguf_file=filename)
# model = AutoModelForCausalLM.from_pretrained('QuantFactory/deepseek-math-7b-rl-GGUF', gguf_file=filename, device_map={"": 0})




In [ ]:
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

In [ ]:
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# state = '\"\"\"Given the Lean 4 tactic state, suggest a next tactic. Do NOT show working"\n\n\
state = 'α : Type u_1\n\
r : α → α → Prop\n\
inst1 : DecidableEq α\n\
inst : IsIrrefl α r\n\
⊢ CutExpand r ≤ InvImage (Finsupp.Lex (cr Π fun x x_1 => x̸ = x_1)\n\
fun x x_1 => x < x_1) ↑toFinsupp[ANSWER]'

tokenized_state = tokenizer(
    state,
    padding="longest",
    max_length=2300,
    truncation=True,
    return_tensors="pt",
)

state_ids = tokenized_state.input_ids.cuda()
state_mask = tokenized_state.attention_mask.cuda()


In [ ]:
with torch.no_grad():
    num_samples = 10
    output = model.generate(
        input_ids=state_ids,
        attention_mask=state_mask,
        max_new_tokens=100,
        # top_k =50,
        # top_p = 0.95,
        num_beams=num_samples,
        do_sample=False,
        # length_penalty=1.0
        num_return_sequences=num_samples,
    )

In [ ]:
from models.end_to_end.tactic_models.causal_generator.model import RetrievalAugmentedGenerator

model = RetrievalAugmentedGenerator.load(
    '../runs/large_lm/deepseek-base-novel-prompt/2024_06_17/17_30_55/checkpoints/last.ckpt', 'cuda', freeze=True)

In [ ]:
import torch

tokenizer = model.tokenizer

In [ ]:
state = ' You are an expert in Lean 3 theorem proving. Suggest a tactic to solve the following goal. Include premises in the following format: <a>premise<\a>, and keep the answer as concise as possible:\n\
α : Type u_1\n\
r : α → α → Prop\n\
inst1 : DecidableEq α\n\
inst : IsIrrefl α r\n\
⊢ CutExpand r ≤ InvImage (Finsupp.Lex (cr Π fun x x_1 => x̸ = x_1)\n\
fun x x_1 => x < x_1) ↑toFinsupp\n\
[ANSWER]'

tokenized_state = tokenizer(
    state,
    padding="longest",
    max_length=2300,
    truncation=True,
    return_tensors="pt",
)

state_ids = tokenized_state.input_ids.cuda()
state_mask = tokenized_state.attention_mask.cuda()

with torch.no_grad():
    num_samples = 10
    output = model.generator.generate(
        input_ids=state_ids,
        attention_mask=state_mask,
        max_new_tokens=200,
        top_k=50,
        top_p=0.95,
        num_beams=num_samples,
        do_sample=True,
        length_penalty=2.0,
        num_return_sequences=num_samples,
    )


In [ ]:
tokenizer.batch_decode(output)

In [ ]:
from models.end_to_end.tactic_models.causal_generator.model import RetrievalAugmentedGenerator

model = RetrievalAugmentedGenerator.load('../runs/large_lm/quant_16_gen/2024_06_24/16_55_13/checkpoints/last.ckpt',
                                         'cuda', freeze=True)


In [ ]:
state = '@[simp] lemma <a>polynomial.leading_coeff_X_sub_C</a> [ring S] (r : S) :\n' \
        '  (X - C r).leading_coeff = 1\n' \
        '\n' \
        '@[simp] lemma <a>polynomial.leading_coeff_X_pow</a> (n : ℕ) : leading_coeff ((X : R[X]) ^ n) = 1\n' \
        '\n' \
        "lemma <a>polynomial.leading_coeff_pow'</a> : leading_coeff p ^ n ≠ 0 →\n" \
        '  leading_coeff (p ^ n) = leading_coeff p ^ n\n' \
        '\n' \
        '@[simp] lemma <a>polynomial.leading_coeff_eq_zero</a> : leading_coeff p = 0 ↔ p = 0\n' \
        '\n' \
        'lemma <a>polynomial.leading_coeff_add_of_degree_eq</a> (h : degree p = degree q)\n' \
        '  (hlc : leading_coeff p + leading_coeff q ≠ 0) :\n' \
        '  leading_coeff (p + q) = leading_coeff p + leading_coeff q\n' \
        '\n' \
        'lemma <a>polynomial.leading_coeff_add_of_degree_lt</a> (h : degree p < degree q) :\n' \
        '  leading_coeff (p + q) = leading_coeff q\n' \
        '\n' \
        'lemma <a>polynomial.leading_coeff_C_mul_X_pow</a> (a : R) (n : ℕ) : leading_coeff (C a * X ^ n) = a\n' \
        '\n' \
        '@[simp] theorem <a>polynomial.leading_coeff_mul_X</a> {p : R[X]} :\n' \
        '  leading_coeff (p * X) = leading_coeff p\n' \
        '\n' \
        '@[simp] lemma <a>polynomial.leading_coeff_quadratic</a> (ha : a ≠ 0) :\n' \
        '  leading_coeff (C a * X ^ 2 + C b * X + C c) = a\n' \
        '\n' \
        '@[simp] lemma <a>polynomial.leading_coeff_X</a> : leading_coeff (X : R[X]) = 1\n' \
        '\n' \
        '@[simp] theorem <a>polynomial.leading_coeff_mul_X_pow</a> {p : R[X]} {n : ℕ} :\n' \
        '  leading_coeff (p * X ^ n) = leading_coeff p\n' \
        '\n' \
        'def <a>polynomial.leading_coeff</a> (p : R[X]) : R := coeff p (nat_degree p)\n' \
        '\n' \
        '@[simp] lemma <a>polynomial.leading_coeff_C</a> (a : R) : leading_coeff (C a) = a\n' \
        '\n' \
        '@[simp] lemma <a>polynomial.monic.leading_coeff</a> {p : R[X]} (hp : p.monic) :\n' \
        '  leading_coeff p = 1\n' \
        '\n' \
        'def <a>polynomial.monic</a> (p : R[X]) := leading_coeff p = (1 : R)\n' \
        '\n' \
        "lemma <a>polynomial.leading_coeff_mul'</a> (h : leading_coeff p * leading_coeff q ≠ 0) :\n" \
        '  leading_coeff (p * q) = leading_coeff p * leading_coeff q\n' \
        '\n' \
        'lemma <a>polynomial.monic.def</a> : monic p ↔ leading_coeff p = 1\n' \
        '\n' \
        'theorem <a>polynomial.leading_coeff_mul_monic</a> {p q : R[X]} (hq : monic q) :\n' \
        '  leading_coeff (p * q) = leading_coeff p\n' \
        '\n' \
        '@[simp] lemma <a>polynomial.leading_coeff_mul</a> (p q : R[X]) : leading_coeff (p * q) =\n' \
        '  leading_coeff p * leading_coeff q\n' \
        '\n' \
        'theorem <a>polynomial.leading_coeff_monic_mul</a> {p q : R[X]} (hp : monic p) :\n' \
        '  leading_coeff (p * q) = leading_coeff q\n' \
        '\n' \
        'R : Type u,\n' \
        '_inst_1 : semiring R,\n' \
        'p q : R[X],\n' \
        'hp : p.monic,\n' \
        'hq : q.monic\n' \
        '⊢ (p * q).monic'


In [ ]:
model.gen_config.strategy = 'beam'

In [ ]:
# output = model.generate(state, None, 8)

In [ ]:
# output

In [ ]:
from experiments.end_to_end.common import zip_strict

num_samples = 8

prompt = ('You are an expert in Lean 3 theorem proving.'
          'Given a set of premises, followed by a goal to prove, suggest a single tactic to solve the goal.'
          'Any premises in the tactic should be included in the following format: <a>premise<\\a>. The goal is: \n\n')



state_with_prompt = [prompt + s + '[ANSWER]' for s in [state]]

tokenized_state = model.tokenizer(
    state_with_prompt,
    padding="longest",
    max_length=1300,
    truncation=True,
    return_tensors="pt",
)

state_ids = tokenized_state.input_ids.cuda()
state_mask = tokenized_state.attention_mask.cuda()


output = model.generator.generate(
    input_ids=state_ids,
    attention_mask=state_mask,
    max_length=1300,
    num_beams=num_samples,
    length_penalty=0.0,
    do_sample=False,
    num_return_sequences=num_samples,
    # early_stopping=False,
    early_stopping=True,
    output_scores=True,
    return_dict_in_generate=True,
   num_beam_groups=8,
    diversity_penalty=10.0,
)


In [ ]:
# Return the output.
raw_output_text = model.tokenizer.batch_decode(
    output.sequences, skip_special_tokens=True
)

raw_scores = output.sequences_scores.tolist()
tactics_with_scores = []

for i in range(len([state])):
    output_text = []
    output_score = []

    for j in range(i * num_samples, (i + 1) * num_samples):
        t = raw_output_text[j]
        t = t.split('[ANSWER]')[-1]
        if t not in output_text:
            output_text.append(t)
            output_score.append(raw_scores[j])

    tactics_with_scores.append(list(zip_strict(output_text, output_score)))

tactics_with_scores

In [ ]:
# model.generator.save_pretrained('../runs/deepseek-base/')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

new_model = AutoModelForCausalLM.from_pretrained('../runs/deepseek-base', device_map='auto', quantization_config=quant_config)


In [ ]:
model.generator = new_model

In [ ]:
model.tokenizer.pad_token

In [ ]:
model.generator.pad_token_id = model.tokenizer.pad_token_id
model.generator.generation_config.pad_token_id = model.tokenizer.pad_token_id
output = model.generate(state, None, 16)
output
